# VYRO Motion Studio — free Kaggle GPU proof

Purpose: prove that a free Kaggle GPU can turn one VYRO storyboard still into a real moving portrait MP4 before attempting the full trailer.

**No paid services.** Run the GPU check first. This notebook does not claim a video exists until the final cell saves and verifies an MP4. If the model download requests Hugging Face access, stop there rather than entering payment details; access may require accepting the model licence on its model page.

In [ ]:
import os, sys, subprocess, platform
print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
    else:
        print('STOP: In Kaggle, open Notebook settings and select a GPU accelerator, then rerun this cell.')
except Exception as e:
    print('PyTorch check failed:', repr(e))
assert 'torch' in sys.modules and torch.cuda.is_available(), 'No GPU detected. Enable a Kaggle GPU accelerator before continuing.'

## 1. Upload one storyboard image

Use the VYRO portrait Beat City / rival-faceoff storyboard image. Upload a PNG or JPG from your phone. The test will use just one image and produce a short clip, not the full 60-second trailer.

In [ ]:
from google.colab import files as _colab_files  # not used on Kaggle; import below is Kaggle-specific
from IPython.display import display
import ipywidgets as widgets
from IPython.display import FileLink
from pathlib import Path
from PIL import Image
import os
from kaggle_secrets import UserSecretsClient  # available on Kaggle; no secrets are required for the basic GPU check
print('For Kaggle, use the notebook file upload control / Add Input to add your storyboard image, then set IMAGE_PATH below.')

### Set the uploaded image path

After uploading/adding the image in Kaggle, replace `IMAGE_PATH` with its actual path (commonly under `/kaggle/input/.../image.png`). If the image is in the notebook session working directory, use that path. The image is fitted to the model's landscape frame internally; we will crop/pad the exported video to portrait without stretching.

In [ ]:
from pathlib import Path
IMAGE_PATH = '/kaggle/input/REPLACE-WITH-YOUR-DATASET/storyboard.png'  # EDIT THIS
OUTPUT_PATH = '/kaggle/working/vyro-motion-test.mp4'
assert Path(IMAGE_PATH).is_file(), f'Image not found: {IMAGE_PATH}. Add your image to the notebook and edit IMAGE_PATH.'
im = Image.open(IMAGE_PATH).convert('RGB')
print('Storyboard image:', im.size, im.mode)
display(im)

## 2. Install video-generation dependencies

This cell installs free open-source Python libraries only. It does not use a paid API.

In [ ]:
%pip -q install -U diffusers transformers accelerate safetensors imageio imageio-ffmpeg

## 3. Generate a short motion proof

This uses Stable Video Diffusion XT image-to-video. Its weights may require accepting Stability AI's model terms on Hugging Face; if the download is denied, do not keep retrying. A Kaggle T4 may need memory optimisations and generation can take several minutes. This is a technical proof, not the final trailer.

In [ ]:
import torch, imageio, numpy as np
from PIL import Image
from diffusers import StableVideoDiffusionPipeline
from diffusers.utils import export_to_video
assert torch.cuda.is_available(), 'GPU is not enabled.'
model_id = 'stabilityai/stable-video-diffusion-img2vid-xt'
pipe = StableVideoDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16, variant='fp16')
pipe.enable_model_cpu_offload()
pipe.unet.enable_forward_chunking()
source = Image.open(IMAGE_PATH).convert('RGB')
# SVD expects 1024x576; preserve the image proportions and crop centrally rather than stretch.
target_ratio = 1024 / 576
w, h = source.size
if w / h > target_ratio:
    nw = int(h * target_ratio); left = (w - nw) // 2; source = source.crop((left, 0, left + nw, h))
else:
    nh = int(w / target_ratio); top = (h - nh) // 2; source = source.crop((0, top, w, top + nh))
source = source.resize((1024, 576), Image.Resampling.LANCZOS)
frames = pipe(source, num_frames=25, num_inference_steps=20, decode_chunk_size=2, motion_bucket_id=127, noise_aug_strength=0.02, generator=torch.manual_seed(42)).frames[0]
landscape_path = '/kaggle/working/vyro-motion-landscape.mp4'
export_to_video(frames, landscape_path, fps=7)
print('Saved:', landscape_path, 'frames:', len(frames))
torch.cuda.empty_cache()

## 4. Convert to portrait without stretching and verify

The generated model output is landscape. This step makes a 9:16 portrait crop, which may cut off parts of the composition; inspect the result before deciding whether the model is suitable for VYRO.

In [ ]:
import subprocess, json, os
cmd = ['ffmpeg','-y','-i','/kaggle/working/vyro-motion-landscape.mp4','-vf',"crop=ih*9/16:ih:(iw-ih*9/16)/2:0,scale=720:1280:flags=lanczos,setsar=1,fps=24,format=yuv420p",'-an','-movflags','+faststart',OUTPUT_PATH]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stderr[-2500:])
assert result.returncode == 0 and os.path.isfile(OUTPUT_PATH) and os.path.getsize(OUTPUT_PATH) > 100_000, 'MP4 export failed or output is suspiciously small.'
probe = subprocess.run(['ffprobe','-v','error','-show_entries','format=duration,size:stream=codec_name,width,height,r_frame_rate','-of','json',OUTPUT_PATH],capture_output=True,text=True,check=True)
info = json.loads(probe.stdout)
print(json.dumps(info, indent=2))
print('VERIFIED OUTPUT:', OUTPUT_PATH)

## 5. Download the proof clip

In Kaggle's right-hand Output panel, download `vyro-motion-test.mp4` from `/kaggle/working/`. Keep this clip separate from the seven previously verified VYRO clips until it has been visually inspected.